# Data Cleaning and Historical Target Definition

This notebook applies the explicit cleaning decisions for the Lending Club data, creates the historical good/bad target, and saves reproducible train/test checkpoints for the following notebooks.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.options.display.max_columns = None
REFERENCE_DATE = pd.Timestamp("2017-12-01")
DATA_PATH = Path("../data/loan_data_2007_2014.csv")
loan_data = pd.read_csv(DATA_PATH, low_memory=False)

## Deterministic conversions and target

The target is historical performance rather than an application-time default label. A value of 0 denotes `Charged Off`, `Default`, `Does not meet the credit policy. Status:Charged Off`, or `Late (31-120 days)`; every other observed status receives 1. This preserves the original historical definition, including its limitation: status availability and loan maturity determine what is observable in this dataset.

In [2]:
loan_data["emp_length_int"] = loan_data["emp_length"].str.extract(r"(\d+)", expand=False).astype("float")
loan_data["term_int"] = loan_data["term"].str.extract(r"(\d+)", expand=False).astype("int")

loan_data["earliest_cr_line_date"] = pd.to_datetime(
    loan_data["earliest_cr_line"], format="%b-%y", errors="coerce"
)
future_credit_line = loan_data["earliest_cr_line_date"] > REFERENCE_DATE
loan_data.loc[future_credit_line, "earliest_cr_line_date"] = (
    loan_data.loc[future_credit_line, "earliest_cr_line_date"] - pd.DateOffset(years=100)
)
loan_data["mths_earliest_cr_line_date"] = np.round(
    (REFERENCE_DATE - loan_data["earliest_cr_line_date"]) / np.timedelta64(1, "D") / 30.4375
)
loan_data["issue_date"] = pd.to_datetime(loan_data["issue_d"], format="%b-%y", errors="coerce")
loan_data["mths_issue_date"] = np.round(
    (REFERENCE_DATE - loan_data["issue_date"]) / np.timedelta64(1, "D") / 30.4375
)

bad_statuses = {
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Late (31-120 days)",
}
loan_data["good_bad"] = (~loan_data["loan_status"].isin(bad_statuses)).astype(int)
status_summary = (
    loan_data.groupby("loan_status", dropna=False)["good_bad"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "good_rate"})
    .sort_values("count", ascending=False)
)
display(status_summary)

,count,good_rate
loan_status,,
Current,224226,1.0
Fully Paid,184739,1.0
Charged Off,42475,0.0
Late (31-120 days),6900,0.0
In Grace Period,3146,1.0
Does not meet the credit policy. Status:Fully Paid,1988,1.0
Late (16-30 days),1218,1.0
Default,832,0.0
Does not meet the credit policy. Status:Charged Off,761,0.0


## Stratified train/test split

The split is created before learning the income-imputation statistic. Stratification preserves the historical target balance in both partitions, so observed good/bad proportions can be compared without an avoidable sampling shift.

In [3]:
inputs = loan_data.drop(columns="good_bad")
target = loan_data["good_bad"]
inputs_train, inputs_test, targets_train, targets_test = train_test_split(
    inputs, target, test_size=0.20, random_state=42, stratify=target
)
print("Train shape:", inputs_train.shape, "Test shape:", inputs_test.shape)
display(pd.DataFrame({
    "count": [targets_train.size, targets_test.size],
    "good_rate": [targets_train.mean(), targets_test.mean()],
}, index=["train", "test"]))

Train shape: (373028, 81) Test shape: (93257, 81)


,count,good_rate
train,373028,0.890695
test,93257,0.890689


## Missingness before repair

The selected fields follow the original cleaning decisions, but the repairs are performed after the split. Missing `total_rev_hi_lim` is replaced by the funded amount from the same row: it is a borrower-specific available amount rather than a generic population statistic. Missing annual income is replaced with the training-partition mean only, then that unchanged value is applied to held-out rows.

For `emp_length_int` and the selected account-event fields, a missing entry is treated as zero recorded years or events. This applies to `acc_now_delinq`, `total_acc`, `pub_rec`, `open_acc`, `inq_last_6mths`, and `delinq_2yrs`. Other fields are retained for subsequent feature analysis, where missingness can itself be useful information.

In [4]:
event_count_fields = [
    "acc_now_delinq",
    "total_acc",
    "pub_rec",
    "open_acc",
    "inq_last_6mths",
    "delinq_2yrs",
]
selected_missingness_fields = [
    "total_rev_hi_lim",
    "annual_inc",
    "emp_length_int",
    *event_count_fields,
]
missing_before = pd.DataFrame({
    "train": inputs_train[selected_missingness_fields].isna().sum(),
    "test": inputs_test[selected_missingness_fields].isna().sum(),
})
display(missing_before)

,train,test
total_rev_hi_lim,56156,14120
annual_inc,4,0
emp_length_int,16793,4215
acc_now_delinq,25,4
total_acc,25,4
pub_rec,25,4
open_acc,25,4
inq_last_6mths,25,4
delinq_2yrs,25,4


In [5]:
annual_inc_train_mean = inputs_train["annual_inc"].mean()

for frame in (inputs_train, inputs_test):
    frame["total_rev_hi_lim"] = frame["total_rev_hi_lim"].fillna(frame["funded_amnt"])
    frame["annual_inc"] = frame["annual_inc"].fillna(annual_inc_train_mean)
    frame["emp_length_int"] = frame["emp_length_int"].fillna(0).astype(int)
    frame[event_count_fields] = frame[event_count_fields].fillna(0)

print(f"Training annual-income mean used for both partitions: {annual_inc_train_mean:,.2f}")

Training annual-income mean used for both partitions: 73,341.89


## Categorical indicators

The original categorical fields are retained and represented with indicators. Indicator definitions are learned from the training partition and applied unchanged to the held-out rows; unseen held-out categories therefore receive zeros across that family.

In [6]:
categorical_fields = [
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "loan_status",
    "purpose",
    "addr_state",
    "initial_list_status",
]

def add_categorical_indicators(frame, columns=None):
    indicators = pd.get_dummies(frame[categorical_fields], prefix=categorical_fields, prefix_sep=":")
    if columns is not None:
        indicators = indicators.reindex(columns=columns, fill_value=False)
    return pd.concat([frame, indicators], axis=1)

inputs_train = add_categorical_indicators(inputs_train)
inputs_test = add_categorical_indicators(inputs_test, columns=inputs_train.columns[len(inputs.columns):])
print(f"Indicator columns: {inputs_train.shape[1] - inputs.shape[1]:,}")

Indicator columns: 125


## Verification and checkpoints

These checks cover the repaired fields and converted columns. Other source fields, including those with potentially informative missingness, remain unchanged for later feature analysis. The saved inputs preserve both the original categorical fields and training-aligned indicators; later model construction must exclude outcome-derived fields such as loan-status indicators to avoid target leakage.

In [7]:
missing_after = pd.DataFrame({
    "train": inputs_train[selected_missingness_fields].isna().sum(),
    "test": inputs_test[selected_missingness_fields].isna().sum(),
})
display(missing_after)
print(inputs_train[["emp_length_int", "term_int", "mths_earliest_cr_line_date", "mths_issue_date"]].dtypes)
assert missing_after.to_numpy().sum() == 0
assert targets_train.index.is_unique and targets_test.index.is_unique
assert set(targets_train.unique()) <= {0, 1}
assert set(targets_test.unique()) <= {0, 1}

,train,test
total_rev_hi_lim,0,0
annual_inc,0,0
emp_length_int,0,0
acc_now_delinq,0,0
total_acc,0,0
pub_rec,0,0
open_acc,0,0
inq_last_6mths,0,0
delinq_2yrs,0,0


emp_length_int                  int64
term_int                        int64
mths_earliest_cr_line_date    float64
mths_issue_date               float64
dtype: object


In [8]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
inputs_train.to_pickle(processed_dir / "clean_inputs_train.pkl")
inputs_test.to_pickle(processed_dir / "clean_inputs_test.pkl")
targets_train.to_pickle(processed_dir / "targets_train.pkl")
targets_test.to_pickle(processed_dir / "targets_test.pkl")

for path in sorted(processed_dir.glob("*.pkl")):
    print(path.name)

clean_inputs_test.pkl
clean_inputs_train.pkl
targets_test.pkl
targets_train.pkl


## Conclusions

The checkpoints preserve an 80/20 stratified historical-performance split. Repair decisions are explicit: same-row funded amount for missing revolving limits, a training-only mean for income, and zero for absent employment length and specified event counts. The target is an educational historical good/bad outcome, not a complete production definition of default risk.

The checkpoints are intentionally intermediate artifacts. They retain information needed for the next feature-engineering notebooks, while the final PD model must use only variables that would be available at the intended decision point.